# HD art v2: The CurioKids Club backgrounds and picture boards

* Remakes the **12 pictures picked from v1** (same prompt + seed = the same picture).
* Adds **7 new backgrounds** for the fun-facts series (night sky, bathroom, kitchen, ocean, river valley, reading nook, windy hills), in portrait and landscape, 2 versions each.
* Adds **7 picture-board close-ups** (landscape, 2 versions each).
* 54 pictures in all, about 1 hour on a T4. Output: **art_v2.zip**.
* Model: **FLUX.1-schnell** (Black Forest Labs, Apache 2.0: free for commercial use). No paid services; nothing is posted anywhere.

**Settings (right-hand panel):** Session options: Accelerator **GPU T4 x2**, Internet **On**. Add-ons, Secrets: **HF_TOKEN** ticked for this notebook.

**Run:** Start session, then **Run All**, and keep the tab open.

**IMPORTANT:** when the last cell shows the contact sheet, download **art_v2.zip** from Output (on the right, click the three dots, then Download). **Stopping the session deletes the files**, so download first, then stop.

In [ ]:
# 1) Install the libraries (about 1-2 minutes)
!pip install -q -U "diffusers>=0.32" "transformers>=4.46" accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# 2) Check the GPU
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 (Session options, right-hand panel) and run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
# T4/P100 have no fast bfloat16, so they use float16; newer GPUs use bfloat16
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("compute type:", DTYPE)

In [ ]:
# 3) The pictures. Keep "no people, no text" in every prompt.
V1_STYLE = ("soft 3D clay-like children's picture-book illustration, rounded friendly shapes, warm pastel colours, "
            "gentle global illumination, soft shadows, shallow depth of field, cheerful and calm, high detail, "
            "no people, no characters, no animals unless named, no text, no letters, no logos, no watermark")
# v2 style: pushes the clay / animated-film look harder, so new pictures match garden / playroom / space
CLAY = ("3D clay render in the style of a modern animated family film, matte clay and plasticine textures, soft rounded shapes, "
        "bright cheerful pastel colours, soft warm global illumination, gentle shadows, slight depth of field, high detail, "
        "no people, no characters, no faces, no text, no letters, no numbers, no logos, no watermark")
BG = "wide open calm ground in the lower part of the picture, interesting details only at the sides and the top"

P, L = "portrait", "landscape"
PICTURES = [
  # --- kept from v1 (exact v1 prompt + the picked seed)
  ("garden",    "a sunny garden with soft grass, round bushes, a few big flowers at the sides, blue sky with fluffy clouds, open empty lawn in the middle", V1_STYLE, [(P, 42), (L, 42)]),
  ("playroom",  "a bright cosy children's playroom, toy shelves and building blocks at the sides, a big round window with daylight, empty soft rug in the middle", V1_STYLE, [(P, 42), (L, 42)]),
  ("space",     "a friendly outer-space scene, deep blue and purple sky full of twinkling stars, a ringed planet and a small moon at the edges, open space in the middle", V1_STYLE, [(P, 11), (L, 11)]),
  ("sprout",    "close-up of a tiny green sprout with two leaves pushing out of dark rich soil, morning sunlight, dew drops, blurred garden behind", V1_STYLE, [(P, 11), (L, 42)]),
  ("waterdrop", "a big shiny water droplet about to fall from a green leaf, sparkling reflections, soft rain and a faint rainbow in a blurred background", V1_STYLE, [(P, 42), (L, 11)]),
  ("earth",     "planet Earth seen from space, bright blue oceans, green land and white clouds, glowing atmosphere, soft starry background, Earth at the lower half", V1_STYLE, [(P, 42), (L, 42)]),
  # --- new backgrounds for the fun-facts series (portrait + landscape, 2 versions)
  ("night_sky",    f"a magical night sky over soft rolling clay hills, thousands of twinkling stars, a glowing crescent moon at the top corner, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("bathroom",     f"a bright clean friendly bathroom, white tiles, a sink with a cup of toothbrushes, a bathtub with bubbles at the side, fluffy towels, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("kitchen",      f"a cosy sunny family kitchen, wooden table, bowl of fruit, glass of water, shelves with jars, a window with daylight, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("ocean",        f"a sunny beach by a calm turquoise sea, gentle waves, soft sand, a few shells, fluffy clouds, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("river_valley", f"a green valley with a winding blue river flowing from far mountains toward the sea, small trees on the banks, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("reading_nook", f"a cosy reading corner with big colourful bookshelves, floor cushions, a warm lamp and alphabet blocks, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  ("windy_hills",  f"green hills on a breezy day, leaves and dandelion seeds blowing in the wind, a kite in the sky, big fluffy clouds, {BG}", CLAY, [(o, s_) for o in (P, L) for s_ in (11, 42)]),
  # --- new picture-board close-ups (landscape only, 2 versions)
  ("soil_layers", "a cut-away cross-section of the ground showing layers: grass on top, dark soil with roots, brown clay, sand and small stones at the bottom", CLAY, [(L, 11), (L, 42)]),
  ("desert",      "a sunny desert with tall green cactus plants, smooth sand dunes and a clear blue sky", CLAY, [(L, 11), (L, 42)]),
  ("teeth",       "a shiny clean white tooth next to a toothbrush with fresh foam and a tube of toothpaste, sparkling", CLAY, [(L, 11), (L, 42)]),
  ("straw_glass", "a clear glass of water with a striped straw in it that looks bent where it enters the water, on a table, side view", CLAY, [(L, 11), (L, 42)]),
  ("apple_cut",   "a red apple cut in half on a wooden board, one half fresh and white, the other half turned brown", CLAY, [(L, 11), (L, 42)]),
  ("moon_day",    "a pale half moon in a bright blue daytime sky above green treetops, soft white clouds", CLAY, [(L, 11), (L, 42)]),
  ("waves",       "big friendly ocean waves rolling toward the shore, white foam, sunlight sparkling on the water", CLAY, [(L, 11), (L, 42)]),
]
SIZES = {P: (864, 1536, (1080, 1920)), L: (1536, 864, (1920, 1080))}   # made at ~1.3 MP, then resized
TOTAL = sum(len(v) for *_, v in PICTURES)
print(TOTAL, "pictures to make")

In [ ]:
# 4) Load FLUX.1-schnell in 4-bit so it fits on one 16 GB GPU (first time downloads ~33 GB; about 5-10 minutes)
import os, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb
MODEL = "black-forest-labs/FLUX.1-schnell"
# Needs the HF_TOKEN Kaggle secret (see the steps at the top)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using the HF_TOKEN secret")
except Exception:
    raise SystemExit("No HF_TOKEN secret: do the one-time Hugging Face step at the top (Add-ons -> Secrets -> HF_TOKEN), then run again.")
q = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
transformer = FluxTransformer2DModel.from_pretrained(MODEL, subfolder="transformer", quantization_config=DiffBnb(**q), torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder_2", quantization_config=TfBnb(**q), torch_dtype=DTYPE)
pipe = FluxPipeline.from_pretrained(MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache()
print("model ready")

In [ ]:
# 5) Make the pictures (about 1 minute each on a T4). Already-made files are skipped, so a re-run continues where it stopped.
import json, time, datetime
import numpy as np
from PIL import Image
OUT = "/kaggle/working/art_v2"
os.makedirs(OUT, exist_ok=True)
manifest = []
t_start = time.time(); n = 0
for name, what, style, jobs in PICTURES:
    prompt = f"{what}, {style}"
    for orient, seed in jobs:
        n += 1
        w, h, final = SIZES[orient]
        fn = f"kids/{name}_{orient}_s{seed}.png"
        path = os.path.join(OUT, fn)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            t = time.time()
            img = pipe(prompt, width=w, height=h, num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256,
                       generator=torch.Generator("cpu").manual_seed(seed)).images[0]
            if np.asarray(img).mean() < 3:
                print("WARNING: black picture for", fn, "- tell Claude")
            img.resize(final, Image.LANCZOS).save(path, optimize=True)
            print(f"[{n}/{TOTAL}] made {fn} in {time.time() - t:.0f}s")
        manifest.append({"file": fn, "name": name, "orientation": orient, "seed": seed, "size": list(final),
                         "made_at": [w, h], "prompt": prompt, "model": MODEL, "steps": 4,
                         "licence": "Apache-2.0 (FLUX.1-schnell)", "date": datetime.date.today().isoformat()})
json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=1)
print(f"done: {len(manifest)} pictures in {(time.time() - t_start) / 60:.0f} min")

In [ ]:
# 6) Contact sheet to look at, and the zip to download (Output -> art_v2.zip)
from PIL import ImageDraw
thumbs = []
for m in manifest:
    im = Image.open(os.path.join(OUT, m["file"])).convert("RGB")
    im.thumbnail((270, 270)); tile = Image.new("RGB", (280, 300), "white"); tile.paste(im, ((280 - im.width) // 2, 5))
    ImageDraw.Draw(tile).text((6, 282), m["file"].split("/")[-1][:40], fill="black"); thumbs.append(tile)
cols = 9; rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 280, rows * 300), "white")
for i, t in enumerate(thumbs): sheet.paste(t, ((i % cols) * 280, (i // cols) * 300))
sheet.save(os.path.join(OUT, "contact_sheet.jpg"), quality=85)
import shutil
shutil.make_archive("/kaggle/working/art_v2", "zip", OUT)
print("zip:", round(os.path.getsize("/kaggle/working/art_v2.zip") / 2**20, 1), "MB")
print("NOW: download art_v2.zip from Output (right-hand panel) BEFORE stopping the session.")
sheet